# ATIS SLU & S&P 500 Experiments

This notebook contains end-to-end experiments for:

- Question 1: RNN-based SLU on ATIS (Slot Filling + Intent Detection)
- Question 2: Recursive time-series forecasting for S&P 500 (Baseline MLP, CNN+LSTM, GRU, TCN)

Follow the sections below to reproduce dataset download, preprocessing, model training and evaluation. Use the scripts under `scripts/` for full training runs.

## Section 1 — Environment Setup & Imports

Install required packages (run in a cell if not installed) and import libraries. Detect device (CUDA / MPS / CPU) and set seeds for reproducibility.

```python
# Install (run once)
# !pip install -r ../requirements.txt

import os
import random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)
```

In [ ]:
# Imports used across the notebook
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.data.preprocess import load_atis_examples, build_vocab, build_label_vocab, ATISDataset, collate_fn
from src.models.baseline import BiRNNSlotFiller, BiLSTMJoint
from src.utils.metrics import slot_f1, slot_classification_report, intent_accuracy

sns.set(style="whitegrid")
print("imports ok")

## Section 2 — Download & Load ATIS Dataset

We use `kagglehub` helper to download the `siddhadev/atis-dataset-clean` dataset and then parse it with our loader.

```python
import kagglehub
path = kagglehub.dataset_download("siddhadev/atis-dataset-clean")
print("Downloaded to:", path)

# load
examples = load_atis_examples(path)
len(examples)
```

In [ ]:
# Try load processed if exists else use raw
proc_dir = Path("../data/processed").resolve()
if proc_dir.exists():
    train = json.load(open(proc_dir / "train.json", "r", encoding="utf8"))
    dev = json.load(open(proc_dir / "dev.json", "r", encoding="utf8"))
    test = json.load(open(proc_dir / "test.json", "r", encoding="utf8"))
    print(f"Loaded processed: train={len(train)} dev={len(dev)} test={len(test)}")
else:
    print("Processed data not found. Run preprocessing script: python -m src.data.preprocess --input data/raw --out data/processed")

---
## Notebook image saving & reproducibility notes ✅

All figures generated in this notebook are saved under the `codes/images/` directory:

- `codes/images/atis/` — images for ATIS experiments
- `codes/images/sp500/` — images for the S&P 500 experiments

Filenames are descriptive (e.g., `birnn_loss.png`, `bilstm_slot_f1.png`, `sp500_acf_close.png`). Use these images directly in your IEEE-format report (high-resolution PNGs are saved).

Run the cells sequentially (restart kernel & run all) to regenerate figures. If long training is needed, prefer running the `scripts/` files and then run the evaluation/plotting cells here to reproduce the figures for the report.

In [ ]:
# --- ATIS: Small BiRNN training demo (kept short for notebook) 
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import islice

proc_dir = Path("../data/processed").resolve()
img_dir = Path("../images/atis").resolve()
img_dir.mkdir(parents=True, exist_ok=True)

if not proc_dir.exists():
    print("Processed data not found. Run preprocessing: python -m src.data.preprocess --input data/raw --out data/processed")
else:
    print("Loading processed ATIS data...")
    train = json.load(open(proc_dir / "train.json","r",encoding="utf8"))
    dev = json.load(open(proc_dir / "dev.json","r",encoding="utf8"))
    test = json.load(open(proc_dir / "test.json","r",encoding="utf8"))
    vocab = json.load(open(proc_dir / "vocab.json","r",encoding="utf8"))

    word2id = vocab['word2id']
    slot2id = vocab['slot2id']
    intent2id = vocab['intent2id']

    # small dataset / loader
    train_ds = ATISDataset(train, word2id, slot2id, intent2id)
    test_ds = ATISDataset(test, word2id, slot2id, intent2id)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

    device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
    print('Device:', device)

    model = BiRNNSlotFiller(len(word2id), embed_dim=128, hidden_dim=128, num_labels=len(slot2id), bidirectional=True, dropout=0.5)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

    # quick training (3 epochs) — limited batches for notebook speed
    train_losses = []
    test_f1s = []
    epochs = 3
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for i, batch in enumerate(islice(train_loader, 200)):
            input_ids = batch['input_ids'].to(device)
            slot_ids = batch['slot_ids'].to(device)
            opt.zero_grad()
            logits = model(input_ids)
            b, t, c = logits.shape
            loss = criterion(logits.view(-1, c), slot_ids.view(-1))
            loss.backward()
            opt.step()
            running += loss.item()
        train_losses.append(running / (i+1))

        # quick eval on test (full)
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                slot_ids = batch['slot_ids'].to(device)
                logits = model(input_ids)
                pred = logits.argmax(dim=-1).cpu().tolist()
                true = slot_ids.cpu().tolist()
                preds.extend(pred)
                trues.extend(true)
        id2slot = {int(v): k for k, v in slot2id.items()}
        pred_labels = [[id2slot.get(pid, 'O') for pid in seq] for seq in preds]
        true_labels = [[id2slot.get(tid, 'O') for tid in seq] for seq in trues]
        f1 = slot_f1(true_labels, pred_labels)
        test_f1s.append(f1)
        print(f"Epoch {epoch+1}/{epochs} — train_loss={train_losses[-1]:.4f} test_f1={f1:.4f}")

    # Save training plots
    plt.figure(figsize=(6,4))
    plt.plot(range(1, epochs+1), train_losses, marker='o')
    plt.title('BiRNN Training Loss (demo)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.tight_layout()
    p = img_dir / 'birnn_train_loss_demo.png'
    plt.savefig(p, dpi=200)
    print('Saved plot to', p)

    plt.figure(figsize=(6,4))
    plt.plot(range(1, epochs+1), test_f1s, marker='o', color='green')
    plt.title('BiRNN Test Slot F1 (demo)')
    plt.xlabel('Epoch')
    plt.ylabel('Slot F1')
    plt.grid(True)
    plt.tight_layout()
    p2 = img_dir / 'birnn_test_f1_demo.png'
    plt.savefig(p2, dpi=200)
    print('Saved plot to', p2)


In [ ]:
# --- S&P 500: Download and Feature Engineering (cyclical encoding) 
import yfinance as yf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

img_dir_sp = Path('../images/sp500').resolve()
img_dir_sp.mkdir(parents=True, exist_ok=True)

print('Downloading ^GSPC (may take a few seconds)...')
df = yf.download('^GSPC', start='2000-01-01', auto_adjust=False)
print('Downloaded rows:', len(df))

# keep OHLC
df = df[['Open','High','Low','Close']].dropna()
# date features
df = df.reset_index()
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day

# cyclical encoding
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)

# Year minmax
df['year_minmax'] = (df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min())

# Save a quick plot of cyclical encodings
plt.figure(figsize=(6,4))
plt.plot(df['Date'].iloc[:365], df['month_sin'].iloc[:365], label='month_sin')
plt.plot(df['Date'].iloc[:365], df['month_cos'].iloc[:365], label='month_cos')
plt.legend()
plt.title('Cyclical Encoding (first year)')
plt.tight_layout()
p = img_dir_sp / 'sp500_cyclical_month.png'
plt.savefig(p, dpi=200)
print('Saved', p)

# Plot ACF and PACF for Close
fig = plot_acf(df['Close'], lags=60)
fig.figure.tight_layout()
fig.figure.savefig(str(img_dir_sp / 'sp500_acf_close.png'), dpi=200)
print('Saved ACF to', img_dir_sp / 'sp500_acf_close.png')
fig2 = plot_pacf(df['Close'], lags=60, method='ywm')
fig2.figure.tight_layout()
fig2.figure.savefig(str(img_dir_sp / 'sp500_pacf_close.png'), dpi=200)
print('Saved PACF to', img_dir_sp / 'sp500_pacf_close.png')

# differencing and replot
close_diff = df['Close'].diff().dropna()
fig3 = plot_acf(close_diff, lags=60)
fig3.figure.tight_layout()
fig3.figure.savefig(str(img_dir_sp / 'sp500_acf_close_diff.png'), dpi=200)
print('Saved differenced ACF to', img_dir_sp / 'sp500_acf_close_diff.png')
